In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
with open(os.path.join(BASE_DIR, "config.json"), "r") as f:
    config = json.load(f)

torch.manual_seed(config["seed"]); np.random.seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR, FEATURES_DIR, RESULTS_DIR = config["paths"]["data_dir"], config["paths"]["features_dir"], config["paths"]["results_dir"]
SPLITS = ["train", "dev", "test"]
TEXT_MODEL, TEXT_DIM = config["text_model"], config["text_feature_dim"]
MAX_LENGTH, BATCH_SIZE = 128, 64
RECOMPUTE = False
SETTINGS = {"model": TEXT_MODEL, "pooling": "mean_over_non_padding_tokens_incl_special", "max_length": MAX_LENGTH, "dim": TEXT_DIM}
META_PATH = os.path.join(FEATURES_DIR, "text_features_meta.json")
print(f"Device: {device} | model: {TEXT_MODEL} | pooling: mean | max_length: {MAX_LENGTH}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)
model = AutoModel.from_pretrained(TEXT_MODEL).to(device).eval()
assert model.config.hidden_size == TEXT_DIM, f"Model hidden size {model.config.hidden_size} != configured text_feature_dim {TEXT_DIM}"
print("RoBERTa model loaded and set to evaluation mode.")

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    """Mean of the token embeddings over non-padding positions: sum(h * mask) / sum(mask)."""
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    return (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

@torch.no_grad()
def encode(texts):
    enc = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
    out = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
    return mean_pool(out.last_hidden_state, enc["attention_mask"]).float().cpu().numpy()

def extract_text_features(df, desc):
    texts = df["Utterance"].astype(str).tolist()
    feats, n_trunc = [], 0
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc=desc):
        batch = texts[i:i + BATCH_SIZE]
        full = tokenizer(batch, add_special_tokens=True, truncation=False)["input_ids"]
        n_trunc += sum(len(x) > MAX_LENGTH for x in full)
        feats.append(encode(batch))
    return np.vstack(feats).astype(np.float32), n_trunc

print("Pooling / extraction helpers defined.")

In [ ]:
meta = json.load(open(META_PATH)) if os.path.exists(META_PATH) else {}
report = {}
for split in SPLITS:
    df = pd.read_csv(os.path.join(DATA_DIR, f"{split}_sent_emo.csv"))
    path = os.path.join(FEATURES_DIR, f"text_features_{split}.npy")
    cached_ok = (not RECOMPUTE and os.path.exists(path) and meta.get(split, {}).get("settings") == SETTINGS
                 and meta[split].get("n_rows") == len(df) and np.load(path, mmap_mode="r").shape == (len(df), TEXT_DIM))
    if cached_ok:
        print(f"[{split}] cached mean-pooled features match the current settings -> skipped")
        report[split] = meta[split]
        continue
    if os.path.exists(path) and meta.get(split, {}).get("settings") != SETTINGS:
        print(f"[{split}] existing cache was produced with different settings (e.g. the old CLS readout) -> recomputing")
    feats, n_trunc = extract_text_features(df, f"Text features ({split})")
    assert feats.shape == (len(df), TEXT_DIM) and np.isfinite(feats).all(), f"[{split}] invalid features {feats.shape}"
    np.save(path, feats)
    report[split] = {"settings": SETTINGS, "n_rows": len(df), "n_truncated": int(n_trunc)}
    meta[split] = report[split]
    json.dump(meta, open(META_PATH, "w"), indent=2)
    print(f"[{split}] saved {feats.shape} to {path} | truncated utterances: {n_trunc} ({n_trunc / len(df):.3%})")

with open(os.path.join(RESULTS_DIR, "text_extraction_report.json"), "w") as f:
    json.dump(report, f, indent=2)
for s, r in report.items():
    if r["n_truncated"] / r["n_rows"] > 0.01:
        print(f"[warn] {s}: more than 1% of utterances exceed {MAX_LENGTH} tokens and were truncated")

In [ ]:
checks = []
def check(name, cond, detail=""):
    checks.append({"test": name, "passed": bool(cond), "detail": str(detail)})
    print(("PASS  " if cond else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))

rs = np.random.RandomState(0)
for split in SPLITS:
    df = pd.read_csv(os.path.join(DATA_DIR, f"{split}_sent_emo.csv"))
    X = np.load(os.path.join(FEATURES_DIR, f"text_features_{split}.npy"))
    check(f"[{split}] shape (N, {TEXT_DIM}) = CSV rows, finite, float32", X.shape == (len(df), TEXT_DIM) and np.isfinite(X).all() and X.dtype == np.float32)
    check(f"[{split}] features vary across utterances (non-degenerate)", X.std(axis=0).mean() > 1e-4, f"mean per-dim std={X.std(axis=0).mean():.4f}")

    idx = rs.choice(len(df), size=min(24, len(df)), replace=False)
    texts = df["Utterance"].astype(str).values
    worst_pad, worst_manual, min_cls_dist = 0.0, 0.0, 1e9
    for i in idx:
        enc = tokenizer([texts[i]], truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)     
        with torch.no_grad():
            h = model(**enc).last_hidden_state[0].float().cpu().numpy()                                          
        manual = h.astype(np.float64).mean(axis=0)                                                               
        worst_pad = max(worst_pad, np.abs(X[i] - encode([texts[i]])[0]).max())
        worst_manual = max(worst_manual, np.abs(X[i] - manual).max())
        min_cls_dist = min(min_cls_dist, np.abs(X[i] - h[0]).max())
    check(f"[{split}] cached (batched, padded) features equal the single-sequence mean of the raw hidden states", worst_manual < 2e-4, f"max |diff|={worst_manual:.1e}")
    check(f"[{split}] features differ from the <s> (CLS) vector", min_cls_dist > 1e-3, f"min max-abs distance={min_cls_dist:.3f}")

meta_now = json.load(open(META_PATH))
check("metadata records mean pooling for every split", all(meta_now[s]["settings"]["pooling"].startswith("mean") for s in SPLITS))
failed = [c for c in checks if not c["passed"]]
json.dump({"n_tests": len(checks), "n_failed": len(failed), "results": checks}, open(os.path.join(RESULTS_DIR, "text_features_verification.json"), "w"), indent=2)
assert not failed, f"{len(failed)} verification check(s) failed: {[c['test'] for c in failed]}"
print(f"\nAll {len(checks)} checks passed.")
print("Phase 3 Text Feature Extraction completed successfully. All text feature caches verified.")